# But du notebook  

Réaliser un entrainement suivie d'un modèle de machine learning sur des données présente dans le dossier data suivie avec le tracking de MLFlow.

# Initialisation

## Packages

In [51]:
import os
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import seaborne as sns

Mise à l'échelle

In [52]:
from sklearn.preprocessing import RobustScaler

Machine learning

In [53]:
# Préporcessing
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# Scorers
from sklearn.metrics import confusion_matrix, make_scorer

# Modeles
from sklearn.linear_model import LogisticRegression

In [61]:
# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV # Recherche d'hyperparametre
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

In [55]:
import mlflow

## Fonctions

In [56]:
def apply_transformation_and_evaluate(y_true, y_pred, metric):
    # Faire attention à l'échelle des données qui sont passer a cette fonction lors du grid search
    
    # Calculer la métrique demandée
    if metric == 'r2':
        return r2_score(y_true_transformed, y_pred_transformed)
        
    elif metric == 'rmse':
        return root_mean_squared_error(y_true_transformed, y_pred_transformed)
        
    elif metric == 'mae':
        return mean_absolute_error(y_true_transformed, y_pred_transformed)
        
    else:
        raise ValueError(f"Il n'y a pas de métrique reconnue pour '{metric}'.")

In [57]:
def create_custom_scorer(metric):
    return make_scorer(lambda y_true, y_pred: apply_transformation_and_evaluate(y_true, y_pred, metric))

In [58]:
# Construire un dictionnaire avec les différents estimateur
scores = {
               "R2" : create_custom_scorer("r2"),
               "MAE" : create_custom_scorer("mae"),
               "RMSE" : create_custom_scorer("rmse")
              }

## code issue de Kaggel

In [59]:
# HOME CREDIT DEFAULT RISK COMPETITION
# Most features are created by applying min, max, mean, sum and var functions to grouped tables. 
# Little feature selection is done and overfitting might be a problem since many features are related.
# The following key ideas were used:
# - Divide or subtract important features to get rates (like annuity and income)
# - In Bureau Data: create specific features for Active credits and Closed credits
# - In Previous Applications: create specific features for Approved and Refused applications
# - Modularity: one function for each table (except bureau_balance and application_test)
# - One-hot encoding for categorical features
# All tables are joined with the application DF using the SK_ID_CURR key (except bureau_balance).
# You can use LightGBM with KFold or Stratified KFold.

# Update 16/06/2018:
# - Added Payment Rate feature
# - Removed index from features
# - Use standard KFold CV (not stratified)

import numpy as np
import pandas as pd
import gc
import time
from contextlib import contextmanager
# from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold, StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print("{} - done in {:.0f}s".format(title, time.time() - t0))

# One-hot encoding for categorical columns with get_dummies
def one_hot_encoder(df, nan_as_category = True):
    original_columns = list(df.columns)
    categorical_columns = [col for col in df.columns if df[col].dtype == 'object']
    df = pd.get_dummies(df, columns= categorical_columns, dummy_na= nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns

# Preprocess application_train.csv and application_test.csv
def application_train_test(num_rows = None, nan_as_category = False):
    # Read data and merge
    df = pd.read_csv('./data/raw/application_train.csv', nrows= num_rows)
    test_df = pd.read_csv('./data/raw/application_test.csv', nrows= num_rows)
    print("Train samples: {}, test samples: {}".format(len(df), len(test_df)))
    df = pd.concat([df, test_df], axis=0, ignore_index=True)

    # Optional: Remove 4 applications with XNA CODE_GENDER (train set)
    drop_index = df.loc[df["CODE_GENDER"] == "XNA"].index
    df = df.drop(drop_index)
    del drop_index
    
    # Categorical features with Binary encode (0 or 1; two categories)
    for bin_feature in ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']:
        df[bin_feature], uniques = pd.factorize(df[bin_feature])
    # Categorical features with One-Hot encode
    df, cat_cols = one_hot_encoder(df, nan_as_category)
    
    # NaN values for DAYS_EMPLOYED: 365.243 -> nan
    # Create an annomalie flag column 
    df['DAYS_EMPLOYED_ANOM'] = 0
    mask = df["DAYS_EMPLOYED"] == 365243
    df.loc[mask, "DAYS_EMPLOYED_ANOM"] = 1
    # Change value
    df.loc[mask, "DAYS_EMPLOYED"] = np.nan
    
    # Some simple new features (percentages)
    df['DAYS_EMPLOYED_PERC'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    df['INCOME_CREDIT_PERC'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['ANNUITY_INCOME_PERC'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    del test_df
    gc.collect()
    return df

# Preprocess bureau.csv and bureau_balance.csv
def bureau_and_balance(num_rows = None, nan_as_category = True):
    bureau = pd.read_csv('./data/raw/bureau.csv', nrows = num_rows)
    bb = pd.read_csv('./data/raw/bureau_balance.csv', nrows = num_rows)
    bb, bb_cat = one_hot_encoder(bb, nan_as_category)
    bureau, bureau_cat = one_hot_encoder(bureau, nan_as_category)
    
    # Bureau balance: Perform aggregations and merge with bureau.csv
    bb_aggregations = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_aggregations[col] = ['mean']
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(bb_aggregations)
    bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns.tolist()])
    bureau = bureau.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau.drop(['SK_ID_BUREAU'], axis=1, inplace= True)
    del bb, bb_agg
    gc.collect()
    
    # Bureau and bureau_balance numeric features
    num_aggregations = {
        'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
        'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
        'DAYS_CREDIT_UPDATE': ['mean'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'AMT_CREDIT_MAX_OVERDUE': ['mean'],
        'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean'],
        'AMT_CREDIT_SUM_LIMIT': ['mean', 'sum'],
        'AMT_ANNUITY': ['max', 'mean'],
        'CNT_CREDIT_PROLONG': ['sum'],
        'MONTHS_BALANCE_MIN': ['min'],
        'MONTHS_BALANCE_MAX': ['max'],
        'MONTHS_BALANCE_SIZE': ['mean', 'sum']
    }
    # Bureau and bureau_balance categorical features
    cat_aggregations = {}
    for cat in bureau_cat: cat_aggregations[cat] = ['mean']
    for cat in bb_cat: cat_aggregations[cat + "_MEAN"] = ['mean']
    
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    bureau_agg.columns = pd.Index(['BURO_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])
    # Bureau: Active credits - using only numerical aggregations
    active = bureau[bureau['CREDIT_ACTIVE_Active'] == 1]
    active_agg = active.groupby('SK_ID_CURR').agg(num_aggregations)
    active_agg.columns = pd.Index(['ACTIVE_' + e[0] + "_" + e[1].upper() for e in active_agg.columns.tolist()])
    bureau_agg = bureau_agg.join(active_agg, how='left', on='SK_ID_CURR')
    del active, active_agg
    gc.collect()
    # Bureau: Closed credits - using only numerical aggregations
    closed = bureau[bureau['CREDIT_ACTIVE_Closed'] == 1]
    closed_agg = closed.groupby('SK_ID_CURR').agg(num_aggregations)
    closed_agg.columns = pd.Index(['CLOSED_' + e[0] + "_" + e[1].upper() for e in closed_agg.columns.tolist()])
    bureau_agg = bureau_agg.join(closed_agg, how='left', on='SK_ID_CURR')
    del closed, closed_agg, bureau
    gc.collect()
    return bureau_agg

# Preprocess previous_applications.csv
def previous_applications(num_rows = None, nan_as_category = True):
    prev = pd.read_csv('./data/raw/previous_application.csv', nrows = num_rows)
    prev, cat_cols = one_hot_encoder(prev, nan_as_category= True)
    # Days 365.243 values -> nan
    prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace= True)
    prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace= True)
    prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace= True)
    prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace= True)
    prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace= True)
    # Add feature: value ask / value received percentage
    prev['APP_CREDIT_PERC'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT']
    # Previous applications numeric features
    num_aggregations = {
        'AMT_ANNUITY': ['min', 'max', 'mean'],
        'AMT_APPLICATION': ['min', 'max', 'mean'],
        'AMT_CREDIT': ['min', 'max', 'mean'],
        'APP_CREDIT_PERC': ['min', 'max', 'mean', 'var'],
        'AMT_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'AMT_GOODS_PRICE': ['min', 'max', 'mean'],
        'HOUR_APPR_PROCESS_START': ['min', 'max', 'mean'],
        'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
        'DAYS_DECISION': ['min', 'max', 'mean'],
        'CNT_PAYMENT': ['mean', 'sum'],
    }
    # Previous applications categorical features
    cat_aggregations = {}
    for cat in cat_cols:
        cat_aggregations[cat] = ['mean']
    
    prev_agg = prev.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
    # Previous Applications: Approved Applications - only numerical features
    approved = prev[prev['NAME_CONTRACT_STATUS_Approved'] == 1]
    approved_agg = approved.groupby('SK_ID_CURR').agg(num_aggregations)
    approved_agg.columns = pd.Index(['APPROVED_' + e[0] + "_" + e[1].upper() for e in approved_agg.columns.tolist()])
    prev_agg = prev_agg.join(approved_agg, how='left', on='SK_ID_CURR')
    # Previous Applications: Refused Applications - only numerical features
    refused = prev[prev['NAME_CONTRACT_STATUS_Refused'] == 1]
    refused_agg = refused.groupby('SK_ID_CURR').agg(num_aggregations)
    refused_agg.columns = pd.Index(['REFUSED_' + e[0] + "_" + e[1].upper() for e in refused_agg.columns.tolist()])
    prev_agg = prev_agg.join(refused_agg, how='left', on='SK_ID_CURR')
    del refused, refused_agg, approved, approved_agg, prev
    gc.collect()
    return prev_agg

# Preprocess POS_CASH_balance.csv
def pos_cash(num_rows = None, nan_as_category = True):
    pos = pd.read_csv('./data/raw/POS_CASH_balance.csv', nrows = num_rows)
    pos, cat_cols = one_hot_encoder(pos, nan_as_category= True)
    # Features
    aggregations = {
        'MONTHS_BALANCE': ['max', 'mean', 'size'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean']
    }
    for cat in cat_cols:
        aggregations[cat] = ['mean']
    
    pos_agg = pos.groupby('SK_ID_CURR').agg(aggregations)
    pos_agg.columns = pd.Index(['POS_' + e[0] + "_" + e[1].upper() for e in pos_agg.columns.tolist()])
    # Count pos cash accounts
    pos_agg['POS_COUNT'] = pos.groupby('SK_ID_CURR').size()
    del pos
    gc.collect()
    return pos_agg
    
# Preprocess installments_payments.csv
def installments_payments(num_rows = None, nan_as_category = True):
    ins = pd.read_csv('./data/raw/installments_payments.csv', nrows = num_rows)
    ins, cat_cols = one_hot_encoder(ins, nan_as_category= True)
    # Percentage and difference paid in each installment (amount paid and installment value)
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    # Days past due and days before due (no negative values)
    ins['DPD'] = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DBD'] = ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']
    ins['DPD'] = ins['DPD'].apply(lambda x: x if x > 0 else 0)
    ins['DBD'] = ins['DBD'].apply(lambda x: x if x > 0 else 0)
    # Features: Perform aggregations
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD': ['max', 'mean', 'sum'],
        'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['max', 'mean', 'sum', 'var'],
        'PAYMENT_DIFF': ['max', 'mean', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    for cat in cat_cols:
        aggregations[cat] = ['mean']
    ins_agg = ins.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index(['INSTAL_' + e[0] + "_" + e[1].upper() for e in ins_agg.columns.tolist()])
    # Count installments accounts
    ins_agg['INSTAL_COUNT'] = ins.groupby('SK_ID_CURR').size()
    del ins
    gc.collect()
    return ins_agg

# Preprocess credit_card_balance.csv
def credit_card_balance(num_rows = None, nan_as_category = True):
    cc = pd.read_csv('./data/raw/credit_card_balance.csv', nrows = num_rows)
    cc, cat_cols = one_hot_encoder(cc, nan_as_category= True)
    # General aggregations
    cc.drop(['SK_ID_PREV'], axis= 1, inplace = True)
    cc_agg = cc.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum', 'var'])
    cc_agg.columns = pd.Index(['CC_' + e[0] + "_" + e[1].upper() for e in cc_agg.columns.tolist()])
    # Count credit card lines
    cc_agg['CC_COUNT'] = cc.groupby('SK_ID_CURR').size()
    del cc
    gc.collect()
    return cc_agg

'''
# LightGBM GBDT with KFold or Stratified KFold
# Parameters from Tilii kernel: https://www.kaggle.com/tilii7/olivier-lightgbm-parameters-by-bayesian-opt/code
def kfold_lightgbm(df, num_folds, stratified = False, debug= False):
    # Divide in training/validation and test data
    train_df = df[df['TARGET'].notnull()]
    test_df = df[df['TARGET'].isnull()]
    print("Starting LightGBM. Train shape: {}, test shape: {}".format(train_df.shape, test_df.shape))
    del df
    gc.collect()
    # Cross validation model
    if stratified:
        folds = StratifiedKFold(n_splits= num_folds, shuffle=True, random_state=1001)
    else:
        folds = KFold(n_splits= num_folds, shuffle=True, random_state=1001)
    # Create arrays and dataframes to store results
    oof_preds = np.zeros(train_df.shape[0])
    sub_preds = np.zeros(test_df.shape[0])
    feature_importance_df = pd.DataFrame()
    feats = [f for f in train_df.columns if f not in ['TARGET','SK_ID_CURR','SK_ID_BUREAU','SK_ID_PREV','index']]
    
    for n_fold, (train_idx, valid_idx) in enumerate(folds.split(train_df[feats], train_df['TARGET'])):
        train_x, train_y = train_df[feats].iloc[train_idx], train_df['TARGET'].iloc[train_idx]
        valid_x, valid_y = train_df[feats].iloc[valid_idx], train_df['TARGET'].iloc[valid_idx]

        # LightGBM parameters found by Bayesian optimization
        clf = LGBMClassifier(
            nthread=4,
            n_estimators=10000,
            learning_rate=0.02,
            num_leaves=34,
            colsample_bytree=0.9497036,
            subsample=0.8715623,
            max_depth=8,
            reg_alpha=0.041545473,
            reg_lambda=0.0735294,
            min_split_gain=0.0222415,
            min_child_weight=39.3259775,
            silent=-1,
            verbose=-1, )

        clf.fit(train_x, train_y, eval_set=[(train_x, train_y), (valid_x, valid_y)], 
            eval_metric= 'auc', verbose= 200, early_stopping_rounds= 200)

        oof_preds[valid_idx] = clf.predict_proba(valid_x, num_iteration=clf.best_iteration_)[:, 1]
        sub_preds += clf.predict_proba(test_df[feats], num_iteration=clf.best_iteration_)[:, 1] / folds.n_splits

        fold_importance_df = pd.DataFrame()
        fold_importance_df["feature"] = feats
        fold_importance_df["importance"] = clf.feature_importances_
        fold_importance_df["fold"] = n_fold + 1
        feature_importance_df = pd.concat([feature_importance_df, fold_importance_df], axis=0)
        print('Fold %2d AUC : %.6f' % (n_fold + 1, roc_auc_score(valid_y, oof_preds[valid_idx])))
        del clf, train_x, train_y, valid_x, valid_y
        gc.collect()

    print('Full AUC score %.6f' % roc_auc_score(train_df['TARGET'], oof_preds))
    # Write submission file and plot feature importance
    if not debug:
        test_df['TARGET'] = sub_preds
        test_df[['SK_ID_CURR', 'TARGET']].to_csv(submission_file_name, index= False)
    display_importances(feature_importance_df)
    return feature_importance_df


# Display/plot feature importance
def display_importances(feature_importance_df_):
    cols = feature_importance_df_[["feature", "importance"]].groupby("feature").mean().sort_values(by="importance", ascending=False)[:40].index
    best_features = feature_importance_df_.loc[feature_importance_df_.feature.isin(cols)]
    plt.figure(figsize=(8, 10))
    sns.barplot(x="importance", y="feature", data=best_features.sort_values(by="importance", ascending=False))
    plt.title('LightGBM Features (avg over folds)')
    plt.tight_layout()
    plt.savefig('lgbm_importances01.png')
'''

def main(debug = False):
    '''
    Appliquer les différentes transformation précédente sur les données de l'étude.

    Return :
        df (dataframe) : dataframe qui contient le résultat des la combinaison et du feature engineering réaliser sur les données. 
    '''
    num_rows = 10000 if debug else None
    df = application_train_test(num_rows)
    with timer("Process bureau and bureau_balance"):
        bureau = bureau_and_balance(num_rows)
        print("Bureau df shape:", bureau.shape)
        df = df.join(bureau, how='left', on='SK_ID_CURR')
        del bureau
        gc.collect()
    with timer("Process previous_applications"):
        prev = previous_applications(num_rows)
        print("Previous applications df shape:", prev.shape)
        df = df.join(prev, how='left', on='SK_ID_CURR')
        del prev
        gc.collect()
    with timer("Process POS-CASH balance"):
        pos = pos_cash(num_rows)
        print("Pos-cash balance df shape:", pos.shape)
        df = df.join(pos, how='left', on='SK_ID_CURR')
        del pos
        gc.collect()
    with timer("Process installments payments"):
        ins = installments_payments(num_rows)
        print("Installments payments df shape:", ins.shape)
        df = df.join(ins, how='left', on='SK_ID_CURR')
        del ins
        gc.collect()
    with timer("Process credit card balance"):
        cc = credit_card_balance(num_rows)
        print("Credit card balance df shape:", cc.shape)
        df = df.join(cc, how='left', on='SK_ID_CURR')
        del cc
        gc.collect()
        
    return df
    
'''   
if __name__ == "__main__":
    submission_file_name = "submission_kernel02.csv"
    with timer("Full model run"):
        main()
'''         

'   \nif __name__ == "__main__":\n    submission_file_name = "submission_kernel02.csv"\n    with timer("Full model run"):\n        main()\n'

# Importation

Chemins des dossiers

In [2]:
input_directory = r"./data/raw/"
out_directory = r"./data/prepared/"

Noms des fichiers

In [3]:
os.listdir(input_directory)

['.ipynb_checkpoints',
 'application_test.csv',
 'application_train.csv',
 'bureau.csv',
 'bureau_balance.csv',
 'credit_card_balance.csv',
 'HomeCredit_columns_description.csv',
 'installments_payments.csv',
 'POS_CASH_balance.csv',
 'previous_application.csv',
 'sample_submission.csv']

In [4]:
files_name = {"test" : "application_test.csv",
              "train" : "application_train.csv"}

Importation

In [5]:
train = pd.read_csv(os.path.join(input_directory, files_name["train"]))
test = pd.read_csv(os.path.join(input_directory, files_name["test"]))

## Importation avec les fonctions de Kaggel

In [85]:
data = main(debug=False)

Train samples: 307511, test samples: 48744
Bureau df shape: (305811, 116)
Process bureau and bureau_balance - done in 18s
Previous applications df shape: (338857, 249)
Process previous_applications - done in 18s
Pos-cash balance df shape: (337252, 18)
Process POS-CASH balance - done in 9s
Installments payments df shape: (339587, 26)
Process installments payments - done in 20s
Credit card balance df shape: (103558, 141)
Process credit card balance - done in 14s


In [86]:
data.shape

(356251, 798)

In [142]:
train = data.loc[~data["TARGET"].isna(), :]
test = data.loc[data["TARGET"].isna(), :]

print(f"Jeux d'entrainement d'une taille de {train.shape}")
print(f"Jeux de test d'une taille de {test.shape}")

Jeux d'entrainement d'une taille de (307507, 798)
Jeux de test d'une taille de (48744, 798)


# Préprocesse

L'étude qualitative des données brut a mis en éviendence un ensemble de transformation qui sont nécéssaire pour réaliser l'entrainement des modèles. Ces différentes transformation sont reprise ici.

## `CODE_GENDER`

## `DAYS_EMPLOYED` 

Une partie des individue présente une valeurs unique, mais étant données que cette feature semble avoir une influence non négligeable sur la target ces valeurs dupliquées on été traité d'une manière particulière.

Il semblerais que les points anormaux présente un taux de défaut plus faible que les autres. Donc cette valeurs semble être signifiante. On a donc envie de la remplacer avec une imputation tous en conservant l'information que les point était anormaux en créant une nouvelle colonne.

La transformation a fonctionner

Il faut aussi faire cette transformation pour le set de test

## Feature engineering

Réaliser dans les fonctions de Kaggel

# Entrainement

## Dataset d'entrée

Des valeurs infinit font planter l'entrainement. Sois issue des transformation, sois des dataset d'origine

In [143]:
# On isole les features explicative de la feature cible et on retire la feature d'id
X = train.loc[:, train.columns[2:]]
y = train.loc[:, "TARGET"]

print(f"X d'une taille de {X.shape}")
print(f"y d'une taille de {y.shape}")

X d'une taille de (307507, 796)
y d'une taille de (307507,)


In [150]:
test = (train == float("inf")).any()

In [151]:
test = test.to_frame().reset_index()

In [152]:
inf_column_list = test.loc[test[0].values == True, "index"].to_list()
inf_column_list 

['PREV_APP_CREDIT_PERC_MAX',
 'PREV_APP_CREDIT_PERC_MEAN',
 'REFUSED_APP_CREDIT_PERC_MAX',
 'REFUSED_APP_CREDIT_PERC_MEAN',
 'INSTAL_PAYMENT_PERC_MAX',
 'INSTAL_PAYMENT_PERC_MEAN',
 'INSTAL_PAYMENT_PERC_SUM']

In [154]:
for column in inf_column_list:
    del_index = train.loc[train[column] == float("inf"), :].index
    train.drop(del_index, inplace=True)

C:\Users\SUZON\AppData\Local\Temp\ipykernel_22088\3325777298.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop(del_index, inplace=True)


In [156]:
(train == float('inf')).any().any()

np.False_

In [157]:
# On isole les features explicative de la feature cible et on retire la feature d'id
X = train.loc[:, train.columns[2:]]
y = train.loc[:, "TARGET"]

print(f"X d'une taille de {X.shape}")
print(f"y d'une taille de {y.shape}")

X d'une taille de (307488, 796)
y d'une taille de (307488,)


## Définition de la métric

La métric dévellopper pondère le nombre de FN et de FP donner par le modèle. L'édée et qu'il est préférable pour la banque de perdre un client qui aurait remboursser (Faux Négatif) que d'accepter un clients qui n'aurais pas rembourssé (Faux Positife). Ainsi on minimise une combinaison linéaire de ces deux mesure telle que score = FN+10*FP. Ce score doit être minimisé.

In [90]:
def my_metric(y_true, y_pred_proba, threshold):
    
    # Formate en un dataframe pour facilité les calculs
    y_true.reset_index(drop=True, inplace=True)
    pred_serie = pd.Series(y_pred_proba, name="pred")
    true_label_serie = pd.Series(y_true, name="true_label")
    df = pd.concat([pred_serie, true_label_serie], axis=1)
    
    # Affectation à une classe en fonction du seuil
    ## Construire une nouvelle colonne
    df.loc[:, "pred_label"] = (df["pred"] >= threshold).astype(int)
    
    # Calcul du score metric
    conf_m = confusion_matrix(df["true_label"], df["pred_label"])
    FN = conf_m[1, 0]
    FP = conf_m[0, 1]
    metric = FN + 10 * FP
    return -metric

# Définir le scorer avec les paramètres supplémentaires
threshold = 0.09
my_scorer = make_scorer(my_metric, response_method="predict_proba", threshold=threshold)


## Définition du pipeline

In [91]:
'''
L'encodage étant fait dans les fonctions issue de Kaggel précédenet cette partie n'est pas utilise

transformer = ColumnTransformer([ # Encodage des variables type object 
              ("Ordi_encoder", OrdinalEncoder(), tow_class_columns),
              ("one-hot_encoder", OneHotEncoder(sparse_output=False), more_class_columns)
                ])
'''
pipe = Pipeline([
              #("transformer", transformer), L'encodage est fait dans les fonctions de Kaggel
              ("imputer", SimpleImputer(strategy = "median")),# Utiliser le KNN ?
              ("Scaler", MinMaxScaler()),
              ("Classifier", LogisticRegression(C = 0.0001))
                ])

## Entrainement

In [160]:
del data

In [ ]:
# Définir les paramètres du GridSearch
param_grid = {'Classifier__C': [0.1, 1, 10]}

# Définir la GridSearch avec le scorer
grid_search = GridSearchCV(pipe, param_grid, scoring=my_scorer, n_jobs=8, error_score='raise')

# Effectuer la recherche
grid_search.fit(X, y)

In [ ]:
# Seuil de 0.9
grid_search.best_score_

In [ ]:
# Seuil de  0.8
grid_search.best_score_

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
best_model

In [ ]:
y_pred = best_model.predict_proba(X)
y_pred

In [ ]:
y_pred = y_pred[:,1]
y_pred

In [ ]:
y_pred = pd.Series(y_pred, name="y_pred")

In [ ]:
pred_df = pd.concat([y.reset_index(drop=True), y_pred], axis=1)

In [ ]:
pred_df.loc[:, "pred_label"] = (pred_df["y_pred"] >= threshold).astype(int)

In [ ]:
pred_df.head()

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import RocCurveDisplay

display = RocCurveDisplay.from_predictions(
    pred_df["pred_label"],
    pred_df["TARGET"],
    name=f"",
    curve_kwargs=dict(color="darkorange"),
    plot_chance_level=True,
    despine=True,
)
_ = display.ax_.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=")",
)

# Autre

## Entrainement

In [13]:
import pprint

In [12]:
from utils import fetch_logged_data

ImportError: cannot import name 'fetch_logged_data' from 'utils' (D:\Users\SUZON\AppData\Local\anaconda3\envs\ml_flow_test\Lib\site-packages\utils\__init__.py)

In [26]:
# Enclencher le suivie
mlflow.set_tracking_uri("sqlite:///data/mlflow.db")
mlflow.sklearn.autolog()
mlflow.set_experiment("Test_UI_iris_dataset")

<Experiment: artifact_location=('file:C:/Users/SUZON/OneDrive - CNR/Documents/Jupyter/Openclassrooms/Projets '
 'Openclassrooms/Projet-7-Implementez-un-modele-de-scoring/mlruns/2'), creation_time=1764254419652, experiment_id='2', last_update_time=1764254419652, lifecycle_stage='active', name='Test_UI_iris_dataset', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [29]:
# faire le grid search
gridsearch.fit(iris.data, iris.target)

2025/11/27 15:53:10 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '072471f1651140ffb9f51abf21629d35', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/11/27 15:53:17 INFO mlflow.sklearn.utils: Logging the 5 best runs, no runs will be omitted.


,estimator,"Pipeline(step...sor', SVC())])"
,param_grid,"{'regressor__C': [2, 4, ...], 'regressor__kernel': ('poly',)}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,with_centering,True


In [19]:
# Fetch des infos
run_id = mlflow.last_active_run().info.run_id

In [21]:
 # show data logged in the child runs
filter_child_runs = f"tags.mlflow.parentRunId = '{run_id}'"
runs = mlflow.search_runs(filter_string=filter_child_runs)
param_cols = [f"params.{p}" for p in parameters.keys()]
metric_cols = ["metrics.mean_test_score"]

In [22]:
print("\n========== child runs ==========\n")
pd.set_option("display.max_columns", None)  # prevent truncating columns
print(runs[["run_id", *param_cols, *metric_cols]])


========== child runs ==========

                             run_id params.regressor__kernel  \
0  9e882bb911d84751a8f39b7bcbddda1c                   linear   
1  cd548001e0fc4322b0896b6c9f70f9d3                   linear   
2  ced1aaef1d33499db44447502599ad86                      rbf   
3  e4e7e12919eb48da8a79a8543691846d                      rbf   

  params.regressor__C  metrics.mean_test_score  
0                   1                 0.960000  
1                  10                 0.960000  
2                   1                 0.953333  
3                  10                 0.960000  
